# Aula 07 — LSA e Busca Híbrida: Vetores Densos pela SVD e Fusão RRF (Teoria)
**Projeto Integrador III — Motor de Busca**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*


In [1]:
# 1. Setup oficial do motor07 e montagem dos 8 documentos do curso
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor07.R")
docs <- docs_aula()          # os 8 documentos do curso
cfg  <- cfg_aula()           # as decisoes canonicas (limiar=2, prf_k=0, rocchio)
ix   <- montar(docs, cfg)    # tudo que as aulas anteriores calcularam
cat("Versao do motor:", MOTOR_VERSAO, "\n")
estado()


[1] "MOTOR_VERSAO: motor07 v1 2026-10-15"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | manter |
| `cfg$stopwords` | character | 0 |  |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao o..., d2 o modelo de espaco vetorial..., ... |
| `ix$tokens` | list | 8 | elementos: d1, d2, d3, d4, d5, d6, d7, d8 |
| `ix$vocab` | character | 45 | a, acelera, aprendizado, avaliacao, bm25, busca, ... |
| `ix$w` | matrix | 45 x 8 | linhas: a, acelera, aprendizado, ... | colunas: d1, d2, d3, d4, ... |


## Módulo 1 — O Descasamento de Vocabulário (*Vocabulary Mismatch*)
Demonstração da falha dos modelos puramente léxicos: duas palavras sinônimas têm cosseno exatamente zero quando não compartilham a grafia exata no vocabulário.


In [1]:
# Funcoes lexicas da Aula 02
cosseno <- function(a, b) {
  den <- sqrt(sum(a^2)) * sqrt(sum(b^2)); if (den == 0) 0 else sum(a*b) / den
}
vq <- function(texto) as.integer(table(factor(tokenizar(texto), levels = ix$vocab))) * ix$idf_tfidf
lexico <- function(texto) sort(apply(ix$w, 2, function(d) cosseno(vq(texto), d)), decreasing = TRUE)

cat("Ranking lexico para 'busca':\n")
round(lexico("busca"), 3)
cat("\nRanking lexico para 'ranqueamento':\n")
round(lexico("ranqueamento"), 3)


Ranking lexico para 'busca':
   d5    d7    d1    d2    d3    d4    d6    d8 
0.270 0.265 0.000 0.000 0.000 0.000 0.000 0.000 

Ranking lexico para 'ranqueamento':
   d3    d1    d2    d4    d5    d6    d7    d8 
0.413 0.000 0.000 0.000 0.000 0.000 0.000 0.000 


## Módulo 2 — Esparso × Denso e a Ideia de Embedding
No espaço TF-IDF de 45 termos, $d_1$ possui apenas 7 dimensões ativas e 38 zeros. No LSA (representação densa), todas as $k$ dimensões contêm valores contínuos aprendidos pelas coocorrências dos dados.


In [1]:
# Analise de esparsidade da matriz de pesos TF-IDF
cat("Dimensoes de w:", paste(dim(ix$w), collapse = " x "), "\n")
cat("Termos com peso nao-nulo em d1:", sum(ix$w[, "d1"] != 0), "de", length(ix$vocab), "\n")


Dimensoes de w: 45 x 8 
Termos com peso nao-nulo em d1: 7 de 45 


## Módulo 3 — LSA: A Decomposição em Valores Singulares (SVD) da TF-IDF
Decomposição $W = U \Sigma V^\top \approx U_k \Sigma_k V_k^\top$. Implementação de `lsa(ix, k)`.


In [1]:
# Funcao canonica do LSA
lsa <- function(ix, k) {
  s <- svd(ix$w)
  U <- s$u[, 1:k, drop = FALSE]; rownames(U) <- ix$vocab
  D <- s$v[, 1:k, drop = FALSE] %*% diag(s$d[1:k], nrow = k)
  rownames(D) <- colnames(ix$w)
  list(U = U, S = s$d[1:k], D = D)
}

s <- svd(ix$w)
cat("Valores singulares sigma_i:\n")
round(s$d, 3)
cat("Energia acumulada por k:\n")
round(cumsum(s$d^2) / sum(s$d^2), 3)

m2 <- lsa(ix, 2)
cat("\nMatriz D de documentos em 2 dimensoes (k=2):\n")
round(m2$D, 3)


Valores singulares sigma_i:
[1] 5.602 5.313 5.022 4.783 4.672 4.522 4.319 4.024
Energia acumulada por k:
[1] 0.170 0.323 0.459 0.583 0.701 0.811 0.912 1.000

Matriz D de documentos em 2 dimensoes (k=2):
     [,1]   [,2]
d1 -0.862 -0.253
d2 -2.417  2.910
d3 -1.340  3.089
d4 -0.583 -0.397
d5 -3.361 -0.685
d6 -0.945  0.428
d7 -3.207 -2.888
d8 -0.434  1.001


## Módulo 4 — Dobrar a Consulta (*Folding-in*), Ranquear e Medir
Projeção $\vec{q}_k = U_k^\top \vec{q}_0$. Ranking com cosseno denso e avaliação de métricas.


In [1]:
# Implementacao do ranking LSA canonico
ranking_lsa <- function(consulta, ix, cfg, modelo) {
  q0 <- vetor_consulta(preparar(consulta, cfg), ix$vocab, ix$idf_tfidf)
  qk <- t(modelo$U) %*% q0
  s  <- apply(modelo$D, 1, function(d) cosseno(qk, d))
  sort(s, decreasing = TRUE)
}

q0 <- vetor_consulta(preparar("modelo de recuperacao", cfg), ix$vocab, ix$idf_tfidf)
qk <- t(m2$U) %*% q0
cat("Consulta dobrada qk (k=2):\n")
round(qk, 3)

r2 <- ranking_lsa("modelo de recuperacao", ix, cfg, m2)
cat("\nRanking LSA (k=2):\n")
round(r2, 3)
cat("\nMetricas LSA (k=2, limiar 2):\n")
g <- gabarito_aula()
round(metricas(r2, g, 2), 3)


Consulta dobrada qk (k=2):
       [,1]
[1,] -0.370
[2,]  0.445

Ranking LSA (k=2):
    d2     d3     d8     d6     d5     d1     d4     d7 
 1.000  0.960  0.960  0.900  0.474  0.397  0.096 -0.039 

Metricas LSA (k=2, limiar 2):
   P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.667     1.000     1.000     1.000     0.966 


## Módulo 5 — Os Limites do LSA: Relações Inventadas em Corpus Pequeno
Quando $k$ é muito baixo em coleções reduzidas, o LSA aproxima documentos que coocorrem com palavras funcionais sem assunto comum.


In [1]:
# Inspecao do ruído em 'ranqueamento'
r_ranq2 <- ranking_lsa("ranqueamento", ix, cfg, m2)
cat("Ranking 'ranqueamento' (k=2):\n")
round(r_ranq2, 3)

m4 <- lsa(ix, 4)
r_ranq4 <- ranking_lsa("ranqueamento", ix, cfg, m4)
cat("\nRanking 'ranqueamento' restaurado com k=4:\n")
round(r_ranq4, 3)


Ranking 'ranqueamento' (k=2):
    d8     d3     d2     d6     d5     d1     d4     d7 
 0.999  0.999  0.949  0.715  0.170  0.086 -0.224 -0.353 

Ranking 'ranqueamento' restaurado com k=4:
    d3     d6     d8     d1     d2     d4     d7     d5 
 0.998  0.720  0.534  0.392  0.317  0.204  0.057 -0.246 


## Módulo 6 — Busca Híbrida: Fusão Reciprocal Rank Fusion (RRF)
A RRF funde posições dos rankings léxico e denso: $\text{RRF}(d) = \sum_r \frac{1}{c + \text{pos}_r(d)}$, $c = 60$.


In [1]:
# Implementacao canonica de RRF
rrf <- function(..., c = 60) {
  rks <- list(...)
  ids <- unique(unlist(lapply(rks, names)))
  s <- sapply(ids, function(d)
    sum(sapply(rks, function(r) {
      p <- which(names(r) == d)
      if (length(p)) 1 / (c + p) else 0 })))
  sort(s, decreasing = TRUE)
}

rb <- ranking_bm25("modelo de recuperacao", ix, cfg)
rh <- rrf(rb, r2)
cat("Escores do Hibrido RRF (c=60):\n")
round(rh, 4)

rc <- ranking_cosseno("modelo de recuperacao", ix, cfg)
cat("\nComparativo de metricas (Cosseno x BM25 x LSA x Hibrido):\n")
round(rbind(cosseno = metricas(rc, g, 2),
            bm25    = metricas(rb, g, 2),
            lsa2    = metricas(r2, g, 2),
            rrf     = metricas(rh, g, 2)), 3)


Escores do Hibrido RRF (c=60):
    d3     d2     d1     d8     d6     d4     d5     d7 
0.0325 0.0323 0.0313 0.0313 0.0308 0.0306 0.0303 0.0294 

Comparativo de metricas (Cosseno x BM25 x LSA x Hibrido):
        P_at_k    AP  RR nDCG_bin nDCG_grad
cosseno  0.333 0.500 0.5    0.651     0.837
bm25     0.667 0.833 1.0    0.920     0.951
lsa2     0.667 1.000 1.0    1.000     0.966
rrf      0.667 1.000 1.0    1.000     0.990
